# Phase 6 — Production API

**Weeks 16–18 · Goal:** Ship DocuMind as a production FastAPI service with guardrails and observability.

## What you will learn

- **FastAPI routes** — `/query`, `/ingest`, `/admin/*`, `/health`, `/metrics`
- **PII redaction** — Microsoft Presidio on queries, answers, and ingest text
- **Rate limiting** — slowapi per-IP limits
- **OpenTelemetry** — distributed traces to Jaeger
- **DocuMind Web** — React frontend on port 5174 proxying `/api` → `rag-api`

> **ASGI app:** `phases.phase_06_production.api.main:app` or `documind.production.app`


In [1]:
import sys
from pathlib import Path

# Run from notebooks/ — project root is one level up
ROOT = Path.cwd()
if not (ROOT / "shared").exists() and (ROOT.parent / "shared").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT.resolve()}")


Project root: C:\Users\dyh\Desktop\RAG


In [2]:
from phases.phase_06_production.api.main import app

routes = sorted({r.path for r in app.routes if hasattr(r, "methods")})
print("Registered routes:")
for path in routes:
    print(f"  {path}")


ModuleNotFoundError: No module named 'slowapi'

In [ ]:
from phases.phase_06_production.guardrails.pii import PIIRedactor

redactor = PIIRedactor()
text = "Contact john.doe@example.com or call 555-123-4567 about Q4 results."
clean, changed = redactor.redact(text)
print("Original:", text)
print("Redacted:", clean)
print("Changed:", changed)


In [ ]:
# Live API smoke test (requires: docker compose --profile production up -d rag-api)
import os
import httpx

API = os.environ.get("RAG_API_URL", "http://localhost:8002")
try:
    r = httpx.get(f"{API}/health", timeout=5.0)
    print(f"Health: {r.status_code} {r.json()}")
    ready = httpx.get(f"{API}/ready", timeout=10.0)
    print(f"Ready:  {ready.status_code} {ready.json()}")
except Exception as exc:
    print(f"API offline at {API}: {exc}")
    print("Start: cd docker && docker compose --profile production up -d qdrant redis rag-api")


## Production stack

| Service | URL | Role |
|---------|-----|------|
| DocuMind Web | http://localhost:5174 | React UI + nginx `/api` proxy |
| RAG API | http://localhost:8002 | FastAPI backend |
| Jaeger | http://localhost:16686 | Trace UI |
| Grafana | http://localhost:3000 | Dashboards (admin/admin) |

## Capstone integration

- **CLI:** `documind ingest --doc ...` or `python -m capstone.cli`
- **Unified imports:** `from documind import RAGPipeline, DocuMind`

You have now walked through all six phases. See [notebooks/README.md](README.md) for setup and [../README.md](../README.md) for Docker quick start.
